# Proof of concept: Adding Spatial Wavelet Decomposition and Reconstruction to Chromatin Deconvolution
February 16, 2024

We've shown that it is possible to decompose a 2D image into wavelet coefficients and reconstruct within cvxpy. The next step is the feasibility with doing a large set of image coefficients and with the chromatin deconvolution. We had previously deconvolved the coefficients of the images, but were unable to enforce valid non-negative coefficients that produce images. So, this may be a chance to find a place where we can enforce valid non-negative f images, because we are now able to reconstruct the images from coefficients within cvxpy to enforce constraints or norms on the minimization function. We will also be checking feasibility and timing of introducing these additional wavelet computations.

In [4]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import numpy as np
import pywt

In [3]:
from cc_src.chromatin_model import ChromatinModel
from src.config import load_yl_replicate1_rg1_alpha_vst_config

config = load_yl_replicate1_rg1_alpha_vst_config()
chromatin_model = ChromatinModel(config)
chromatin_model.load_mnase_gene('CLB2', replicate=1)

Loading MNase reads for CLB2...Done.


In [6]:
chromatin_model.create_deconvolution_bins(bin_width=16, bin_height=16,
                                         prom_len=256, gb_len=512)

Unflattened the input data is of shape: (16, 16, 48)
The size of our input data, G is: (16, 768)


In [7]:
chromatin_model.setup_deconv_model()

deconv_model = chromatin_model.deconv_model
deconv_model.gamma = 0.001
H = deconv_model.H

image_shape = chromatin_model.deconv_hist_unflattened.shape[1:]
G = chromatin_model.G

In [ ]:
from src.deconvolve_wavelet_chromatin import deconvolve_wavelet_chromatin

f, rn, sn = deconvolve_wavelet_chromatin(deconv_model, H, G,
     verbose=True, image_shape=image_shape)
chromatin_model.f = f
chromatin_model.rn = rn
chromatin_model.sn = sn


                                     CVXPY                                     
                                     v1.4.1                                    
(CVXPY) Feb 20 09:33:30 AM: Your problem has 174336 variables, 1 constraints, and 0 parameters.
(CVXPY) Feb 20 09:33:30 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 20 09:33:30 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 20 09:33:30 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 20 09:33:30 AM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 20 09:33:30 AM: Compiling problem (target solver=MOSEK).

In [ ]:
pred_Gs = H@f
pred_G_imgs = pred_Gs.reshape((-1, *image_shape))
f_imgs = f.reshape((-1, *image_shape))

In [ ]:
for i in range(0, 200, 25):
    plt.figure(figsize=(1.5, 0.5))
    plt.imshow(f_imgs[i], cmap='magma_r', origin='lower')
    plt.show()

In [ ]:
for i in range(pred_G_imgs.shape[0]):
    plt.imshow(pred_G_imgs[i], cmap='magma_r', origin='lower')
    plt.show()